# MedNeXt MEN-RT — Smoke Test (50 cases · Fold 0: 30 ep · Folds 1–4: 20 ep)

**Before running:**
1. Attach the BraTS-MEN-RT dataset as an input (right panel → Add Data)
2. Enable **Internet** (right panel → Internet → ON) — needed to clone the repo
3. Enable **GPU** (right panel → Accelerator → GPU T4 or P100)
4. Run cells **top to bottom**

**To RESUME a fold after disconnection:** Add `--continue-training` to the relevant training cell.

**Pipeline stages:**
```
GPU check → Install deps → Clone repo → Validate → Prepare (50 cases)
→ Preprocess → Splits → Install trainer
→ Train Fold 0 (30 ep) → Train Folds 1–4 (20 ep each)
→ Export CSVs → Plots → Overlays → Prompts
```

In [ ]:
# ─────────────────────────────────────────────
# Cell 1: GPU check
# Verify Kaggle has assigned a GPU before starting the expensive pipeline.
# If you see 'No GPU' — go to Settings → Accelerator → GPU.
# ─────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ─────────────────────────────────────────────
# Cell 2: Install Python dependencies
# nnunet==1.7.1 is the nnU-Net version that MedNeXt is built on.
# MedNeXt is installed from GitHub (official repo, no fork).
# ─────────────────────────────────────────────
!pip install -q pyyaml simpleitk scipy pandas matplotlib tensorboard
!pip install -q nnunet==1.7.1
!pip install -q git+https://github.com/MIC-DKFZ/MedNeXt.git
print('Installation complete.')

In [ ]:
# ─────────────────────────────────────────────
# Cell 3: Clone the thesis code repository
# The repo contains all pipeline scripts and the config file.
# If the repo already exists (re-run scenario), pull the latest changes.
# ─────────────────────────────────────────────
import os
REPO_DIR = '/kaggle/working/brats-menrt-mednext-thesis'

if os.path.exists(REPO_DIR):
    print('Repo already exists — pulling latest changes...')
    os.system(f'cd {REPO_DIR} && git pull origin main')
else:
    print('Cloning repository...')
    os.system('cd /kaggle/working && git clone https://github.com/maryampervaiz-alt/brats-menrt-mednext-thesis.git')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

In [ ]:
# ─────────────────────────────────────────────
# Cell 4: Verify dataset accessibility
# Confirms the BraTS-MEN-RT dataset is mounted at the expected Kaggle path.
# The paths must match configs/mednext_nnunet.yaml → train_root / val_root.
# ─────────────────────────────────────────────
import os

TRAIN_ROOT = '/kaggle/input/datasets/maryampervaiz24029/brats-men-rt/BraTS2024-MEN-RT-TrainingData/BraTS-MEN-RT-Train-v2'
VAL_ROOT   = '/kaggle/input/datasets/maryampervaiz24029/brats-men-rt/BraTS2024-MEN-RT-ValidationData/BraTS-MEN-RT-Val-v1'

train_ok = os.path.exists(TRAIN_ROOT)
val_ok   = os.path.exists(VAL_ROOT)

print(f'Train root exists : {train_ok}  ({len(os.listdir(TRAIN_ROOT)) if train_ok else "—"} cases)')
print(f'Val root exists   : {val_ok}   ({len(os.listdir(VAL_ROOT))   if val_ok   else "—"} cases)')

if not train_ok or not val_ok:
    raise FileNotFoundError(
        'Dataset not found at expected path.\n'
        'Run: !find /kaggle/input -maxdepth 4 -type d\n'
        'then update configs/mednext_nnunet.yaml → train_root / val_root.'
    )

print('\nDataset verified — ready to proceed.')

In [ ]:
# ─────────────────────────────────────────────
# Cell 5: Validate full setup
# Checks dataset paths, case counts, CLI availability, trainer importability.
# Read the JSON output carefully — it shows effective_train_case_count = 50.
# ─────────────────────────────────────────────
os.chdir(REPO_DIR)
!python scripts/validate_mednext_nnunet_setup.py \
    --config configs/mednext_nnunet.yaml \
    --check-trainer

In [ ]:
# ─────────────────────────────────────────────
# Cell 6: Prepare dataset
# Copies 50 stratified-by-label-volume train cases and 20 val cases
# into the nnUNet raw data format (imagesTr / labelsTr / imagesTs).
# Writes dataset.json and subset_manifest.json.
# ─────────────────────────────────────────────
os.chdir(REPO_DIR)
!python scripts/run_mednext_nnunet.py \
    --config configs/mednext_nnunet.yaml \
    --mode prepare

In [ ]:
# ─────────────────────────────────────────────
# Cell 7: Plan and preprocess
# nnUNet automatically analyses the dataset intensity distribution,
# and configures optimal patch size, batch size, normalisation, and
# data augmentation.  Do NOT override these values — nnUNet's auto-
# configuration is one of its key strengths.
# ─────────────────────────────────────────────
os.chdir(REPO_DIR)
!python scripts/run_mednext_nnunet.py \
    --config configs/mednext_nnunet.yaml \
    --mode preprocess

In [ ]:
# ─────────────────────────────────────────────
# Cell 8: Create stratified 5-fold cross-validation splits
# Writes splits_final.pkl into nnUNet_preprocessed so that each fold's
# train/val partition is balanced across tumour volume strata.
# ─────────────────────────────────────────────
os.chdir(REPO_DIR)
!python scripts/run_mednext_nnunet.py \
    --config configs/mednext_nnunet.yaml \
    --mode make-splits

In [ ]:
# ─────────────────────────────────────────────
# Cell 9: Install custom MedNeXt trainer
# Creates a thin wrapper class (nnUNetTrainerV2_MedNeXt_S_kernel3_MENRT)
# that reads MEDNEXT_MAX_EPOCHS at runtime.  This lets the same trainer
# name be used across different epoch budgets without editing source files.
# ─────────────────────────────────────────────
os.chdir(REPO_DIR)
!python scripts/run_mednext_nnunet.py \
    --config configs/mednext_nnunet.yaml \
    --mode install-trainer

## Training

nnUNet saves checkpoints automatically:
- `model_best.model` — highest validation Dice so far
- `model_latest.model` — most recent epoch (for resuming)
- `model_final_checkpoint.model` — end of training

**If Kaggle disconnects mid-fold:** re-run the same cell with `--continue-training` added.

In [ ]:
# ─────────────────────────────────────────────
# Cell 10: Train Fold 0  —  30 epochs
# Fold 0 is the primary evaluation fold.  It gets more epochs
# because its best model is used for qualitative analysis and
# for generating foundation-model prompts.
#
# TO RESUME if interrupted:
#   add --continue-training  to the command below.
# ─────────────────────────────────────────────
os.chdir(REPO_DIR)
!python scripts/run_mednext_nnunet.py \
    --config configs/mednext_nnunet.yaml \
    --mode train \
    --fold 0 \
    --max-epochs 30

In [ ]:
# ─────────────────────────────────────────────
# Cell 11: Train Fold 1  —  20 epochs
# TO RESUME: add --continue-training
# ─────────────────────────────────────────────
os.chdir(REPO_DIR)
!python scripts/run_mednext_nnunet.py \
    --config configs/mednext_nnunet.yaml \
    --mode train \
    --fold 1 \
    --max-epochs 20

In [ ]:
# ─────────────────────────────────────────────
# Cell 12: Train Fold 2  —  20 epochs
# TO RESUME: add --continue-training
# ─────────────────────────────────────────────
os.chdir(REPO_DIR)
!python scripts/run_mednext_nnunet.py \
    --config configs/mednext_nnunet.yaml \
    --mode train \
    --fold 2 \
    --max-epochs 20

In [ ]:
# ─────────────────────────────────────────────
# Cell 13: Train Fold 3  —  20 epochs
# TO RESUME: add --continue-training
# ─────────────────────────────────────────────
os.chdir(REPO_DIR)
!python scripts/run_mednext_nnunet.py \
    --config configs/mednext_nnunet.yaml \
    --mode train \
    --fold 3 \
    --max-epochs 20

In [ ]:
# ─────────────────────────────────────────────
# Cell 14: Train Fold 4  —  20 epochs
# TO RESUME: add --continue-training
# ─────────────────────────────────────────────
os.chdir(REPO_DIR)
!python scripts/run_mednext_nnunet.py \
    --config configs/mednext_nnunet.yaml \
    --mode train \
    --fold 4 \
    --max-epochs 20

## Post-Training Analysis
The following cells generate all figures and prompt files.  
They are safe to re-run at any time after training.

In [ ]:
# ─────────────────────────────────────────────
# Cell 15: Export training results to CSV
# Reads nnUNet summary.json from each completed fold and writes:
#   mednext_fold_manifest.csv   — one row per fold, mean metrics
#   mednext_per_case_metrics.csv — one row per validation case
# These CSVs are required by plot_results.py (next cell).
# ─────────────────────────────────────────────
os.chdir(REPO_DIR)
!python scripts/export_mednext_results.py \
    --config configs/mednext_nnunet.yaml

In [ ]:
# ─────────────────────────────────────────────
# Cell 16: Generate publication-ready figures
# Produces (in menrt_repo_artifacts/figures/):
#   fold_metrics_bar.png         — grouped bar, one group per fold
#   per_case_dice_boxplot.png    — Dice distribution across folds
#   per_case_hd95_boxplot.png    — HD95 distribution across folds
#   dice_vs_hd95_scatter.png     — Dice vs HD95 per case
#   fold_metrics_table.csv       — mean ± SD table for thesis
#   best_worst_cases.csv         — top-5 and bottom-5 cases by Dice
# ─────────────────────────────────────────────
os.chdir(REPO_DIR)
!python scripts/plot_results.py \
    --config configs/mednext_nnunet.yaml \
    --dpi 300

In [ ]:
# ─────────────────────────────────────────────
# Cell 17: Generate segmentation overlay figures (fold 0)
# Creates axial / coronal / sagittal overlay panels for 6 cases
# from fold 0 validation set.  If best_worst_cases.csv exists,
# it preferentially selects the best and worst cases.
# ─────────────────────────────────────────────
import os
os.chdir(REPO_DIR)

# Path to best_worst_cases.csv from the previous plotting step
import yaml
with open('configs/mednext_nnunet.yaml') as f:
    _cfg = yaml.safe_load(f)['mednext_nnunet']
_bw_csv = f"{_cfg['results_folder']}/menrt_repo_artifacts/figures/best_worst_cases.csv"

import subprocess, sys
cmd = [
    sys.executable, 'scripts/visualize_predictions.py',
    '--config', 'configs/mednext_nnunet.yaml',
    '--fold', '0',
    '--mode', 'val',
    '--num-cases', '6',
    '--dpi', '200',
]
if os.path.exists(_bw_csv):
    print(f'Using best_worst_cases.csv: {_bw_csv}')
    cmd += ['--best-worst-csv', _bw_csv]
else:
    print('best_worst_cases.csv not found — visualising first 6 cases.')

subprocess.run(cmd, check=True)

In [ ]:
# ─────────────────────────────────────────────
# Cell 18: Generate foundation-model prompts from fold-0 predictions
# Converts MedNeXt validation predictions into structured JSON prompts
# (bounding box, centroid, positive/negative sample points) for
# SAM-Med3D / MedSAM / SAM2 refinement.
# This is the core input to the second stage of the thesis pipeline.
# ─────────────────────────────────────────────
os.chdir(REPO_DIR)
!python scripts/mednext_to_prompts.py \
    --config configs/mednext_nnunet.yaml \
    --source val \
    --fold 0 \
    --n-pos 5 \
    --n-neg 5 \
    --neg-dilation-mm 5.0

In [ ]:
# ─────────────────────────────────────────────
# Cell 19: List all output artifacts
# Kaggle automatically saves everything under /kaggle/working.
# This cell gives a tree view so you know what to download.
# ─────────────────────────────────────────────
import yaml, os
with open(f'{REPO_DIR}/configs/mednext_nnunet.yaml') as f:
    _cfg = yaml.safe_load(f)['mednext_nnunet']

results_base = _cfg['results_folder']
artifacts    = os.path.join(results_base, 'menrt_repo_artifacts')

print('=== Checkpoint files ===')
for fold in range(5):
    fold_dir = os.path.join(
        results_base, 'nnUNet', _cfg['network'], _cfg['task_name'],
        f"{_cfg['trainer_name']}__{_cfg['plans_identifier']}", f'fold_{fold}'
    )
    for ckpt in ['model_best.model', 'model_latest.model', 'model_final_checkpoint.model']:
        p = os.path.join(fold_dir, ckpt)
        status = 'OK' if os.path.exists(p) else 'missing'
        print(f'  fold_{fold}/{ckpt}: {status}')

print('\n=== Reports ===')
reports_dir = os.path.join(artifacts, 'reports')
if os.path.exists(reports_dir):
    for f in sorted(os.listdir(reports_dir)):
        print(f'  {f}')

print('\n=== Figures ===')
figs_dir = os.path.join(artifacts, 'figures')
if os.path.exists(figs_dir):
    for root, dirs, files in os.walk(figs_dir):
        for fname in sorted(files):
            rel = os.path.relpath(os.path.join(root, fname), figs_dir)
            print(f'  {rel}')

print('\n=== Prompts ===')
prompts_dir = os.path.join(artifacts, 'prompts')
if os.path.exists(prompts_dir):
    for root, dirs, files in os.walk(prompts_dir):
        print(f'  {os.path.relpath(root, prompts_dir)}: {len(files)} file(s)')

In [ ]:
# ─────────────────────────────────────────────
# Cell 20: (Optional) Display training curves inline
# nnUNet saves progress.png for each fold.  Display them here for
# a quick sanity check of the loss curves before downloading.
# ─────────────────────────────────────────────
import yaml, os
from IPython.display import Image, display
with open(f'{REPO_DIR}/configs/mednext_nnunet.yaml') as f:
    _cfg = yaml.safe_load(f)['mednext_nnunet']

for fold in range(5):
    progress_png = os.path.join(
        _cfg['results_folder'], 'nnUNet', _cfg['network'], _cfg['task_name'],
        f"{_cfg['trainer_name']}__{_cfg['plans_identifier']}",
        f'fold_{fold}', 'progress.png'
    )
    if os.path.exists(progress_png):
        print(f'\nFold {fold} training curves:')
        display(Image(filename=progress_png, width=700))
    else:
        print(f'Fold {fold}: progress.png not found (fold may not have completed).')